# Browsing SIDM coffea outputs on EOS

Every merged `.coffea` written to the shared SIDM area
(`/store/group/lpcmetx/SIDM/coffea_outputs/<producer>/<study>/`) carries a
`.meta.yaml` sidecar recording the selections, histogram collections, input
samples, and provenance that produced it (see `sidm.tools.metadata`). This
notebook crawls those sidecars into a single index, so you can find an
output -- your own or a collaborator's -- by selection, histogram, sample,
input file, producer, study, commit, schema, or date, and recover the
`root://` path to fetch it.

**Requirements.** Run on LPC or EAF with the `sidm_venv` kernel and a valid
Kerberos ticket (check with `klist`); the crawl reads EOS over xrootd. No
VOMS proxy is needed to read.

In [1]:
import pathlib
import sys

# Make the sidm package importable when running from sidm/test_notebooks/.
repo_root = pathlib.Path.cwd()
while not (repo_root / "sidm").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from sidm.tools import output_browser as ob

## Build the index

`build_index()` lists every sidecar under the shared area and reads them in
parallel. Pass `producers=[...]` to restrict the crawl to specific users, or
`cache="index.json"` to reuse a previous crawl. The cache is not
auto-invalidated, so pass `refresh=True` to re-read after new outputs have
been written.

In [2]:
df = ob.build_index(verbose=True)
len(df)

indexed 7 outputs from 7 sidecars under /store/group/lpcmetx/SIDM/coffea_outputs


7

## The full index at a glance

`summary()` collapses the per-output lists into a compact table. The full,
unabridged columns -- the input ROOT file lists, per-sample cross sections,
the derived `coffea_url`, and so on -- stay on `df`; see `ob.INDEX_COLUMNS`.

In [3]:
ob.summary(df)

,producer,study,name,n_samp,selections,hist_colls,n_hists,created,size,commit
0,mjose,lpc_dask_multi_v1,lpc_dask_multi,2,"base, baseNoLj","muon_base, electron_base",36,2026-06-15T13:28:48Z,57.9 KB,d082590d
1,murtazas,lifetime_study,lifetime_fullgrid,180,baseNoLj_noTrigger,genA_lifetime,13,2026-06-03T02:05:27Z,1.1 MB,799be0e0
2,murtazas,smoke_v1,2Mu2E_100GeV_0p25GeV_0p2mm,1,"base, baseNoLj","muon_base, electron_base",32,2026-05-28T16:50:08Z,21.2 KB,547f6cec
3,murtazas,smoke_v1,2Mu2E_500GeV_1p2GeV_1p9mm,1,"base, baseNoLj","muon_base, electron_base",32,2026-05-28T16:50:08Z,34.5 KB,547f6cec
4,murtazas,lpc_dask_multi_v1,lpc_dask_multi,2,"base, baseNoLj","muon_base, electron_base",32,2026-05-28T16:37:48Z,54.8 KB,547f6cec
5,murtazas,lpc_dask_2Mu2E_500GeV_1p2GeV_1p9mm_v1,lpc_dask_2Mu2E_500GeV_1p2GeV_1p9mm,1,base,muon_base,13,2026-05-28T15:31:45Z,20.3 KB,547f6cec
6,murtazas,smoke_v4,2Mu2E_500GeV_1p2GeV_1p9mm,1,base,muon_base,13,2026-05-28T15:29:53Z,20.3 KB,547f6cec


## Interactive browser

`browser(df)` renders filter controls over the indexed fields above a live
results table, and an **Inspect an output** dropdown that shows a selected
output's full metadata and a ready-to-run `xrdcp` command. The widgets are
interactive only with a live kernel -- run this cell here to use them.

In [ ]:
ob.browser(df)

## Programmatic queries

The same filters are available without the widgets. `query()` returns the
matching rows. All filters are case-insensitive substring matches: scalar
filters (`producer`, `study`, `commit`, `schema`, `coffea_version`) match
the column, and list filters (`selection`, `hist`, `sample`, `input_file`,
`hist_collection`) match a row if any element contains the query. Different
filters are combined with **AND** -- a row is kept only if it satisfies
every filter passed -- while a list passed to a single filter is ORed.

In [5]:
hits = ob.query(df, hist="muon_dxy", selection="base")
ob.summary(hits)

,producer,study,name,n_samp,selections,hist_colls,n_hists,created,size,commit
0,mjose,lpc_dask_multi_v1,lpc_dask_multi,2,"base, baseNoLj","muon_base, electron_base",36,2026-06-15T13:28:48Z,57.9 KB,d082590d
1,murtazas,smoke_v1,2Mu2E_100GeV_0p25GeV_0p2mm,1,"base, baseNoLj","muon_base, electron_base",32,2026-05-28T16:50:08Z,21.2 KB,547f6cec
2,murtazas,smoke_v1,2Mu2E_500GeV_1p2GeV_1p9mm,1,"base, baseNoLj","muon_base, electron_base",32,2026-05-28T16:50:08Z,34.5 KB,547f6cec
3,murtazas,lpc_dask_multi_v1,lpc_dask_multi,2,"base, baseNoLj","muon_base, electron_base",32,2026-05-28T16:37:48Z,54.8 KB,547f6cec
4,murtazas,lpc_dask_2Mu2E_500GeV_1p2GeV_1p9mm_v1,lpc_dask_2Mu2E_500GeV_1p2GeV_1p9mm,1,base,muon_base,13,2026-05-28T15:31:45Z,20.3 KB,547f6cec
5,murtazas,smoke_v4,2Mu2E_500GeV_1p2GeV_1p9mm,1,base,muon_base,13,2026-05-28T15:29:53Z,20.3 KB,547f6cec


In [6]:
# Recover the path to fetch one output.
row = hits.iloc[0]
print(ob.xrdcp_command(row))            # fetch the .coffea
print(ob.xrdcp_command(row, which="meta"))  # fetch the .meta.yaml sidecar
# ob.load_coffea(row) stages the .coffea to a temp dir and returns the loaded object.

xrdcp -f root://cmseos.fnal.gov//store/group/lpcmetx/SIDM/coffea_outputs/mjose/lpc_dask_multi_v1/lpc_dask_multi.coffea .
xrdcp -f root://cmseos.fnal.gov//store/group/lpcmetx/SIDM/coffea_outputs/mjose/lpc_dask_multi_v1/lpc_dask_multi.meta.yaml .


## How outputs land here

The dask-from-notebook and Condor pipelines save each `.coffea` and its
`.meta.yaml` sidecar to
`/store/group/lpcmetx/SIDM/coffea_outputs/<your-username>/<study>/`. Anything
written there by any collaborator appears in this index on the next crawl.